# MLP Input-Output Mapping

How the MLP transforms its input: linearity, selectivity,
transformation structure, and output diversity.

## Why This Matters

MLP input output mapping provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_input_output_mapping import (
    mlp_linearity_measure, mlp_input_selectivity,
    mlp_transformation_structure, mlp_output_diversity,
    mlp_mapping_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## MLP Linearity

How linear is the MLP's input-output relationship?

In [ ]:
for layer in range(4):
    result = mlp_linearity_measure(model, tokens, layer=layer)
    flag = ' [LINEAR]' if result['is_linear'] else ''
    print(f"  Layer {layer}: mean_cos={result['mean_cosine']:.4f}{flag}")
    for p in result['per_position']:
        print(f"    Pos {p['position']}: cos={p['input_output_cosine']:.4f}, "
              f"in={p['input_norm']:.4f}, out={p['output_norm']:.4f}")

## Input Selectivity

Does the MLP respond differently to different inputs?

In [ ]:
for layer in range(4):
    result = mlp_input_selectivity(model, tokens, layer=layer)
    flag = ' [SELECTIVE]' if result['is_selective'] else ''
    print(f"  Layer {layer}: mean={result['mean_norm']:.4f}, "
          f"CV={result['coefficient_of_variation']:.4f}{flag}")

## Transformation Structure

Amplification, rotation, and direction changes.

In [ ]:
for layer in range(4):
    result = mlp_transformation_structure(model, tokens, layer=layer)
    flag = ' [AMPLIFYING]' if result['is_amplifying'] else ''
    print(f"  Layer {layer}: dir_change={result['mean_direction_change']:.4f}, "
          f"norm_change={result['mean_norm_change']:.4f}{flag}")

## Output Diversity

In [ ]:
for layer in range(4):
    result = mlp_output_diversity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    print(f"  Layer {layer}: output_sim={result['mean_output_similarity']:.4f}{flag}")

## Mapping Summary

In [ ]:
result = mlp_mapping_summary(model, tokens)
for p in result['per_layer']:
    flags = []
    if p['is_linear']: flags.append('LINEAR')
    if p['is_selective']: flags.append('SELECTIVE')
    flag_str = f" [{', '.join(flags)}]" if flags else ''
    print(f"  Layer {p['layer']}: cos={p['mean_cosine']:.4f}, "
          f"CV={p['selectivity_cv']:.4f}{flag_str}")